<a href="https://colab.research.google.com/github/karenlyuan/TIS_Report2/blob/main/TIS_Report2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

In [3]:
# データの読み込み / Load the dataset
file_path = 'job_placement.csv'
df = pd.read_csv(file_path)

gpa_col = 'gpa'
univ_col = 'college_name'
target_col = 'placement_status'



# 文字データを数値に変換 / Convert text data to numerical data
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col].astype(str))



# 特徴量（X）と正解ラベル（y）の分割 / Split into feartures(X) and target label(y)
X = df.drop(columns=[target_col, 'salary'])
y = df[target_col]
# salaryは採用された場合にしかデータがないので、特徴量として使うとスコアが100%になる。そのため、salaryのところは使わない。
# Dropped "salary", because it only exists for placed candidates and score would be 100%



# 学習用とテスト用に分割 / Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=28)



# モデルの学習と正解率を計算する関数 / Function to train the model and calculate accuracy
def get_accuracy(X_train_data, X_test_data):
    model = RandomForestClassifier(random_state=42, n_estimators=100)
    model.fit(X_train_data, y_train)
    predictions = model.predict(X_test_data)
    return accuracy_score(y_test, predictions)

# 検証 / Execute
# パターン1：すべての特徴量を使う / Pattern1: use all features
acc_full = get_accuracy(X_train, X_test)

# パターン2: 大学名を消す / Pattern2: remove college name
X_train_no_univ = X_train.drop(columns=[univ_col])
X_test_no_univ = X_test.drop(columns=[univ_col])
acc_no_univ = get_accuracy(X_train_no_univ, X_test_no_univ)

# パターン3: GPAを消す : Pattern3: remove GPA
X_train_no_gpa = X_train.drop(columns=[gpa_col])
X_test_no_gpa = X_test.drop(columns=[gpa_col])
acc_no_gpa = get_accuracy(X_train_no_gpa, X_test_no_gpa)



drop_univ = acc_full - acc_no_univ
drop_gpa = acc_full - acc_no_gpa

print("Accuracy with all features",acc_full)
print("Accuracy without college name",acc_no_univ)
print("Accuracy drop without college name",drop_univ)
print("Accuracy without GPA",acc_no_gpa)
print("Accuracy drop without GPA",drop_gpa)

Accuracy with all features 0.9642857142857143
Accuracy without college name 0.8
Accuracy drop without college name 0.16428571428571426
Accuracy without GPA 0.9571428571428572
Accuracy drop without GPA 0.0071428571428571175


In [4]:
print("Dataset shape:", df.shape)
print("Test samples:", len(y_test))
print("Full accuracy:", acc_full)
print("Without college:", acc_no_univ)
print("Without GPA:", acc_no_gpa)

Dataset shape: (700, 11)
Test samples: 140
Full accuracy: 0.9642857142857143
Without college: 0.8
Without GPA: 0.9571428571428572


In [5]:
# Statistical Hypothesis Testing
## Generate predictions for all three models
def get_predictions(X_train_data, X_test_data):
    model = RandomForestClassifier(
        random_state=42,
        n_estimators=100
    )

    model.fit(X_train_data, y_train)
    predictions = model.predict(X_test_data)

    return predictions


# Generate predictions using the same test samples

# Model 1: All features
pred_full = get_predictions(X_train, X_test)

# Model 2: Without college name
pred_no_univ = get_predictions(
    X_train_no_univ,
    X_test_no_univ
)

# Model 3: Without GPA
pred_no_gpa = get_predictions(
    X_train_no_gpa,
    X_test_no_gpa
)



## Implement McNemar's test
from scipy.stats import binomtest

def mcnemar_test(y_true, pred_a, pred_b):

    # Identify correct and incorrect predictions
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)

    # Model A correct, Model B incorrect
    b = sum(correct_a & ~correct_b)

    # Model A incorrect, Model B correct
    c = sum(~correct_a & correct_b)

    # Calculate exact two-sided McNemar p-value
    if b + c == 0:
        p_value = 1.0
    else:
        p_value = binomtest(
            min(b, c),
            n=b + c,
            p=0.5,
            alternative='two-sided'
        ).pvalue

    print("Model A correct, Model B incorrect:", b)
    print("Model A incorrect, Model B correct:", c)
    print("P-value:", p_value)

    if p_value < 0.05:
        print("Reject H0: Significant difference")
    else:
        print("Fail to reject H0: No significant difference")

    return p_value


  ## Perform the two statistical tests
  # Experiment 1: Effect of removing college name

print("McNemar Test: Full Model vs. Without College Name")

p_college = mcnemar_test(
    y_test,
    pred_full,
    pred_no_univ
)


# Experiment 2: Effect of removing GPA

print("\nMcNemar Test: Full Model vs. Without GPA")

p_gpa = mcnemar_test(
    y_test,
    pred_full,
    pred_no_gpa
)


## Adjust for multiple comparisons
from statsmodels.stats.multitest import multipletests

p_values = [p_college, p_gpa]

adjusted = multipletests(
    p_values,
    alpha=0.05,
    method='holm'
)

print("\nResults after Holm correction:")

print("College name adjusted p-value:", adjusted[1][0])
print("GPA adjusted p-value:", adjusted[1][1])

print("College name significant:", adjusted[0][0])
print("GPA significant:", adjusted[0][1])


McNemar Test: Full Model vs. Without College Name
Model A correct, Model B incorrect: 24
Model A incorrect, Model B correct: 1
P-value: 1.5497207641601562e-06
Reject H0: Significant difference

McNemar Test: Full Model vs. Without GPA
Model A correct, Model B incorrect: 2
Model A incorrect, Model B correct: 1
P-value: 1.0
Fail to reject H0: No significant difference

Results after Holm correction:
College name adjusted p-value: 3.0994415283203125e-06
GPA adjusted p-value: 1.0
College name significant: True
GPA significant: False
